# NovelForge - Note 2 : Deep Learning Fondamental

Objectif : construire un modele recurrent de type LSTM pour predire plusieurs genres a partir des descriptions nettoyees.

Ce notebook couvre les jalons 5, 6 et 7 : justification du choix RNN/LSTM, strategie contre le vanishing gradient, entrainement avec Adam/EarlyStopping et comparaison avec la baseline TF-IDF.

## 1. Imports et configuration

La definition du modele, la tokenisation, le padding, les DataLoaders et les boucles d'entrainement sont encapsules dans `src/dl_models.py`. Le notebook orchestre l'experience.

In [1]:
from pathlib import Path
import os
import sys
import time

import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    PROJECT_DIR = cwd.parent
else:
    PROJECT_DIR = cwd

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from dl_models import (
    LSTMGenreClassifier,
    LSTMTrainingConfig,
    TextVocabulary,
    compute_pos_weight,
    evaluate_lstm_model,
    find_best_threshold,
    get_device,
    make_dataloader,
    train_lstm_model,
)
from preprocessing import (
    TextPreprocessor,
    add_filtered_label_column,
    clean_dataframe,
    drop_columns_if_present,
    infer_column,
    parse_multilabel_cell,
    remove_synopsis_anomalies,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

device = get_device()
print(f"Device used: {device}")
PROJECT_DIR

Device used: cpu


WindowsPath('C:/Users/ClémentPERRET/OneDrive - EQUATERRE-VDS/Bureau/Cours/DeepLearning')

## 2. Chargement et nettoyage des donnees

On reutilise la logique de preprocessing des jalons precedents : suppression de la colonne technique `cover`, nettoyage de `description`, filtrage des `tags` dans une taxonomie de genres, puis suppression des textes trop courts.

In [2]:
def load_dataset(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix in {".json", ".jsonl"}:
        return pd.read_json(path, lines=suffix == ".jsonl")
    raise ValueError(f"Unsupported dataset format: {suffix}")


GENRE_VOCABULARY = [
    "Action",
    "Adventure",
    "Comedy",
    "Drama",
    "Fantasy",
    "Romance",
    "School Life",
    "Slice of Life",
    "Supernatural",
    "Mystery",
    "Psychological",
    "Horror",
    "Historical",
    "Sci Fi",
    "Sports",
    "Martial Arts",
    "Magic",
    "Isekai",
    "Harem",
    "Mecha",
    "Demons",
    "Seinen",
    "Shoujo",
    "Shounen",
    "Josei",
    "BL",
    "GL",
    "Yaoi",
    "Yuri",
    "Shounen-ai",
    "Shoujo-ai",
    "Smut",
    "Wuxia",
    "Xianxia",
]

raw_candidates = [
    Path(os.getenv("NOVELFORGE_DATASET", "")) if os.getenv("NOVELFORGE_DATASET") else None,
    PROJECT_DIR / "data" / "data.csv",
    PROJECT_DIR / "data" / "dataset.csv",
]
raw_path = next((path for path in raw_candidates if path is not None and path.exists()), None)
if raw_path is None:
    raise FileNotFoundError("No dataset found. Place data.csv in data/ or set NOVELFORGE_DATASET.")

df_raw = load_dataset(raw_path)
colonnes_a_supprimer = ["cover"]
df_raw = drop_columns_if_present(df_raw, colonnes_a_supprimer)

text_column = infer_column(df_raw.columns, ["synopsis", "summary", "description", "overview", "plot", "resume"])
label_column = infer_column(df_raw.columns, ["genres", "genre", "tags", "categories", "labels", "target"])

df_raw = add_filtered_label_column(df_raw, label_column, "genre_labels", GENRE_VOCABULARY)
preprocessor = TextPreprocessor(lowercase=True, remove_urls=True, lemmatize=True)
df = clean_dataframe(df_raw, text_column=text_column, clean_column="synopsis_clean", preprocessor=preprocessor)
df = remove_synopsis_anomalies(df, clean_column="synopsis_clean", min_words=5)
df = df[df["genre_labels"].str.len().gt(0)].copy()

print(f"Dataset: {raw_path}")
print(f"Rows after cleaning: {len(df):,}")
print(f"Text column: {text_column}")
print(f"Label column: genre_labels")
display(df[["title", text_column, "synopsis_clean", "genre_labels"]].head())

Dataset: C:\Users\ClémentPERRET\OneDrive - EQUATERRE-VDS\Bureau\Cours\DeepLearning\data\data.csv
Rows after cleaning: 69,511
Text column: description
Label column: genre_labels


,title,description,synopsis_clean,genre_labels
0,Salad Days (Tang LiuZang) - Part 2,The second season of Salad Days (Tang LiuZang).,the second season of salad day tang liuzang,"[BL, Romance, Shounen-ai, Sports]"
1,The Master of Diabolism,"As the grandmaster who founded the Demonic Sect, Wei WuXian roamed the world in his wanton ways, hated by millions for the chaos he crea...",as the grandmaster who found the demonic sect wei wuxian roam the world in his wanton way hat by million for the chao he creat in the en...,"[Action, Adventure, BL, Comedy, Mystery, Romance, Shounen-ai, Martial Arts, Supernatural, Xianxia]"
2,JoJo's Bizarre Adventure Part 7: Steel Ball Run,"Set in 1890, Steel Ball Run spotlights Gyro Zepelli and Johnny Joestar as they pit their spirits on a Fifty Million Dollar race across t...",set in steel ball run spotlight gyro zepelli and johnny joestar as they pit their spirit on a fifty million dollar race across the heart...,"[Action, Adventure, Horror, Mystery, Seinen, Historical]"
3,A Sign of Affection,"Yuki is a typical college student, whose world revolves around her friends, social media, and the latest sales. But when a chance encoun...",yuki is a typical college student whose world revolve around her friend social media and the latest sale but when a chance encounter on ...,"[Romance, Shoujo, Slice of Life]"
4,Moriarty the Patriot,"Before he was Sherlock’s rival, Moriarty fought against the unfair class caste system in London by making sure corrupt nobility got thei...",before he was sherlock s rival moriarty fought against the unfair class caste system in london by mak sure corrupt nobility got their co...,"[Mystery, Shounen, Historical]"


## Jalon 5 - Pourquoi un LSTM plutot qu'un modele TF-IDF ?

La baseline TF-IDF transforme chaque description en vecteur de frequences ponderees. Cette representation est efficace et rapide, mais elle perd l'ordre des mots : les textes "the hero saves the villain" et "the villain saves the hero" peuvent partager presque les memes composantes TF-IDF alors que leur structure semantique change.

Un RNN traite le texte comme une sequence \((x_1, x_2, ..., x_T)\). A chaque pas de temps, il met a jour un etat cache \(h_t = f(x_t, h_{t-1})\). Le modele peut donc apprendre que certains mots prennent un sens different selon leur contexte precedent. Pour NovelForge, c'est important car les genres dependent souvent de motifs narratifs sequentiels : reincarnation puis monde parallele, romance scolaire, enquete surnaturelle, progression martiale, etc.

Le LSTM est choisi car il conserve explicitement une memoire de sequence via un etat de cellule \(c_t\). Contrairement a TF-IDF, il peut modeliser des dependances entre tokens eloignes dans le synopsis, ce qui est pertinent pour detecter des genres qui ne sont pas portes par un seul mot-cle.

## 3. Encodage multilabel et split train/validation/test

Les genres filtres sont transformes en matrice binaire. On separe ensuite les donnees en train, validation et test. La validation sert a l'EarlyStopping et au choix d'hyperparametres.

In [3]:
TEXT_COLUMN = "synopsis_clean"
LABEL_COLUMN = "genre_labels"

model_df = df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
model_df[LABEL_COLUMN] = model_df[LABEL_COLUMN].apply(parse_multilabel_cell)
model_df = model_df[model_df[LABEL_COLUMN].str.len().gt(0)].copy()

# Keep the same genre frequency policy as the baseline notebook.
genre_counts = model_df[LABEL_COLUMN].explode().value_counts()
MIN_GENRE_FREQ = 50
kept_genres = set(genre_counts[genre_counts >= MIN_GENRE_FREQ].index)
model_df[LABEL_COLUMN] = model_df[LABEL_COLUMN].apply(lambda labels: [label for label in labels if label in kept_genres])
model_df = model_df[model_df[LABEL_COLUMN].str.len().gt(0)].copy()

# On CPU, a full LSTM run can be slow. GPU keeps the full dataset; CPU uses a reproducible subset.
MAX_ROWS_CPU = 12_000
if device.type == "cpu" and len(model_df) > MAX_ROWS_CPU:
    model_df = model_df.sample(n=MAX_ROWS_CPU, random_state=42).copy()

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(model_df[LABEL_COLUMN]).astype("float32")
X = model_df[TEXT_COLUMN].fillna("").astype(str)

X_train_text, X_test_text, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
X_train_text, X_valid_text, y_train, y_valid = train_test_split(X_train_text, y_train, test_size=0.15, random_state=42)

print(f"Rows used for DL: {len(model_df):,}")
print(f"Train: {len(X_train_text):,} | Valid: {len(X_valid_text):,} | Test: {len(X_test_text):,}")
print(f"Number of labels: {len(mlb.classes_)}")
display(genre_counts.head(20).to_frame("count"))

Rows used for DL: 12,000
Train: 8,160 | Valid: 1,440 | Test: 2,400
Number of labels: 34


,count
genre_labels,
Romance,29762
Comedy,21282
Drama,18702
Fantasy,16125
Action,12734
BL,12667
School Life,12582
Yaoi,10131
Seinen,9235


## 4. Tokenisation, vocabulaire et padding

Le LSTM attend des sequences d'identifiants entiers. Le vocabulaire est appris uniquement sur le train pour eviter toute fuite d'information. Les sequences sont tronquees ou completees par du padding a longueur fixe.

In [4]:
base_config = LSTMTrainingConfig(
    embedding_dim=128,
    hidden_dim=96,
    num_layers=1,
    dropout=0.35,
    bidirectional=True,
    learning_rate=1e-3,
    batch_size=512,
    epochs=3,
    patience=1,
    threshold=0.2,
    max_length=160,
    max_vocab_size=20_000,
    min_freq=2,
)

vocabulary = TextVocabulary(max_vocab_size=base_config.max_vocab_size, min_freq=base_config.min_freq)
vocabulary.fit(X_train_text)

X_train_seq = vocabulary.transform(X_train_text, max_length=base_config.max_length)
X_valid_seq = vocabulary.transform(X_valid_text, max_length=base_config.max_length)
X_test_seq = vocabulary.transform(X_test_text, max_length=base_config.max_length)

print(f"Vocabulary size: {vocabulary.size:,}")
print(f"Padded sequence shape: {X_train_seq.shape}")

Vocabulary size: 11,728
Padded sequence shape: (8160, 160)


## Jalon 6 - Vanishing Gradient, LSTM, activations et Adam

Un RNN simple applique une recurrence du type \(h_t = 	anh(Wx_t + Uh_{t-1})\). Lors de la retropropagation dans le temps, le gradient est multiplie de nombreuses fois par les matrices de transition et par les derivees des activations. Si ces produits sont inferieurs a 1, le gradient tend vers 0 : c'est le vanishing gradient. Le modele apprend alors mal les dependances longues.

Le LSTM reduit ce probleme grace a ses portes : forget gate, input gate et output gate. La cellule \(c_t\) agit comme un chemin de memoire plus stable. La forget gate decide quelle information conserver, l'input gate decide quoi ajouter, et l'output gate decide quoi exposer a l'etat cache. Les sigmoides des gates produisent des valeurs entre 0 et 1, tandis que \(	anh\) controle l'echelle des contenus. Cette architecture permet de conserver des signaux importants sur plusieurs dizaines de tokens.

L'optimiseur Adam est utilise car il combine momentum et taux d'apprentissage adaptatif par parametre. Il stabilise l'apprentissage sur des donnees textuelles sparse et bruitees. L'EarlyStopping surveille le F1 micro de validation : si le modele cesse de progresser, l'entrainement s'arrete avant de surapprendre. La recherche d'hyperparametres compare ici quelques combinaisons de `hidden_dim`, `dropout` et `learning_rate`.

Comme les genres sont desequilibres, la loss utilise aussi un `pos_weight` par label : les erreurs sur les classes rares pesent davantage. Enfin, le seuil multilabel n?est pas impose a 0.5; il est ajuste sur la validation pour maximiser le F1 micro, ce qui est plus adapte a un probleme multilabel sparse.

## 5. Recherche d'hyperparametres et entrainement

On teste un petit nombre de configurations pour rester raisonnable en temps de calcul. Le meilleur modele est selectionne sur le F1 micro validation.

In [5]:
train_loader_base = make_dataloader(X_train_seq, y_train, batch_size=base_config.batch_size, shuffle=True)
positive_class_weight = compute_pos_weight(y_train)
valid_loader_base = make_dataloader(X_valid_seq, y_valid, batch_size=base_config.batch_size, shuffle=False)
test_loader_base = make_dataloader(X_test_seq, y_test, batch_size=base_config.batch_size, shuffle=False)

hyperparameter_candidates = [
    base_config,
    LSTMTrainingConfig(
        embedding_dim=128,
        hidden_dim=64,
        num_layers=1,
        dropout=0.25,
        bidirectional=True,
        learning_rate=8e-4,
        batch_size=512,
        epochs=3,
        patience=1,
        threshold=0.2,
        max_length=160,
        max_vocab_size=20_000,
        min_freq=2,
    ),
]

search_results = []
best_model = None
best_config = None
best_history = None
best_valid_f1 = -1.0
training_start = time.perf_counter()

for index, config in enumerate(hyperparameter_candidates, start=1):
    print(f"Training candidate {index}/{len(hyperparameter_candidates)}: {config}")
    model = LSTMGenreClassifier(
        vocab_size=vocabulary.size,
        num_labels=len(mlb.classes_),
        embedding_dim=config.embedding_dim,
        hidden_dim=config.hidden_dim,
        num_layers=config.num_layers,
        dropout=config.dropout,
        bidirectional=config.bidirectional,
    )
    model, history = train_lstm_model(
        model,
        train_loader_base,
        valid_loader_base,
        config=config,
        device=device,
        pos_weight=positive_class_weight,
    )
    valid_metrics = evaluate_lstm_model(model, valid_loader_base, threshold=config.threshold, target_names=list(mlb.classes_), device=device)
    result = {
        "candidate": index,
        "hidden_dim": config.hidden_dim,
        "dropout": config.dropout,
        "learning_rate": config.learning_rate,
        "epochs_ran": len(history),
        "valid_f1_micro": valid_metrics["f1_micro"],
        "valid_f1_macro": valid_metrics["f1_macro"],
        "total_seconds": sum(row["epoch_seconds"] for row in history),
    }
    search_results.append(result)

    if valid_metrics["f1_micro"] > best_valid_f1:
        best_valid_f1 = valid_metrics["f1_micro"]
        best_model = model
        best_config = config
        best_history = history

total_training_seconds = time.perf_counter() - training_start
search_df = pd.DataFrame(search_results).sort_values("valid_f1_micro", ascending=False)
display(search_df.round(4))
print(f"Total DL training/search time: {total_training_seconds:.1f} seconds")
print(f"Best config: {best_config}")

Training candidate 1/2: LSTMTrainingConfig(embedding_dim=128, hidden_dim=96, num_layers=1, dropout=0.35, bidirectional=True, learning_rate=0.001, batch_size=512, epochs=3, patience=1, threshold=0.2, max_length=160, max_vocab_size=20000, min_freq=2)
Training candidate 2/2: LSTMTrainingConfig(embedding_dim=128, hidden_dim=64, num_layers=1, dropout=0.25, bidirectional=True, learning_rate=0.0008, batch_size=512, epochs=3, patience=1, threshold=0.2, max_length=160, max_vocab_size=20000, min_freq=2)


,candidate,hidden_dim,dropout,learning_rate,epochs_ran,valid_f1_micro,valid_f1_macro,total_seconds
0,1,96,0.35,0.0010,2,0.1713,0.158,248.5706
1,2,64,0.25,0.0008,2,0.1713,0.158,134.0717


Total DL training/search time: 392.2 seconds
Best config: LSTMTrainingConfig(embedding_dim=128, hidden_dim=96, num_layers=1, dropout=0.35, bidirectional=True, learning_rate=0.001, batch_size=512, epochs=3, patience=1, threshold=0.2, max_length=160, max_vocab_size=20000, min_freq=2)


## 6. Evaluation du meilleur LSTM

On evalue le meilleur candidat sur le test avec les memes metriques que la baseline TF-IDF : F1 micro, F1 macro, F1 weighted, Jaccard samples et Hamming loss.

In [6]:
valid_proba = evaluate_lstm_model(
    best_model,
    valid_loader_base,
    threshold=best_config.threshold,
    target_names=list(mlb.classes_),
    device=device,
)["y_proba"]
best_threshold, best_valid_threshold_f1 = find_best_threshold(y_valid, valid_proba, average="micro")

print(f"Best validation threshold: {best_threshold:.2f}")
print(f"Validation F1 micro at best threshold: {best_valid_threshold_f1:.4f}")

test_metrics = evaluate_lstm_model(
    best_model,
    test_loader_base,
    threshold=best_threshold,
    target_names=list(mlb.classes_),
    device=device,
)

summary = pd.Series(
    {
        "dl_test_f1_micro": test_metrics["f1_micro"],
        "dl_test_f1_macro": test_metrics["f1_macro"],
        "dl_test_f1_weighted": test_metrics["f1_weighted"],
        "dl_test_jaccard_samples": test_metrics["jaccard_samples"],
        "dl_test_hamming_loss": test_metrics["hamming_loss"],
        "best_threshold": best_threshold,
        "dl_training_seconds": total_training_seconds,
    }
).to_frame("score")

display(summary.round(4))
print(test_metrics["classification_report_text"])

Best validation threshold: 0.45
Validation F1 micro at best threshold: 0.1717


,score
dl_test_f1_micro,0.1718
dl_test_f1_macro,0.1588
dl_test_f1_weighted,0.3145
dl_test_jaccard_samples,0.0931
dl_test_hamming_loss,0.8670
best_threshold,0.4500
dl_training_seconds,392.1650


               precision    recall  f1-score   support

       Action       0.19      1.00      0.31       445
    Adventure       0.12      0.82      0.21       239
           BL       0.18      1.00      0.30       428
       Comedy       0.31      1.00      0.48       748
       Demons       0.02      1.00      0.04        48
        Drama       0.26      1.00      0.41       620
      Fantasy       0.23      1.00      0.37       546
           GL       0.03      1.00      0.05        63
        Harem       0.02      1.00      0.05        58
   Historical       0.06      0.99      0.11       148
       Horror       0.03      1.00      0.05        67
       Isekai       0.02      0.79      0.05        52
        Josei       0.07      1.00      0.14       176
        Magic       0.03      1.00      0.05        62
 Martial Arts       0.02      0.83      0.05        47
        Mecha       0.01      1.00      0.02        18
      Mystery       0.05      1.00      0.09       116
Psycholog

### Sauvegarde des artefacts LSTM pour Streamlit

Cette cellule persiste le meilleur LSTM, son vocabulaire, ses labels et son seuil optimal. L'application Streamlit peut ainsi afficher l'approche Deep Learning fondamentale sans reentrainer le modele a chaque lancement.

In [7]:
import joblib

MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

LSTM_MODEL_PATH = MODELS_DIR / "lstm_novelforge.pt"
LSTM_METADATA_PATH = MODELS_DIR / "lstm_metadata.joblib"

if best_model is None or best_config is None:
    raise RuntimeError("Aucun modele LSTM entraine a sauvegarder. Relancer la cellule d'entrainement.")

best_config.threshold = float(best_threshold)
best_model_cpu = best_model.to("cpu")
torch.save(best_model_cpu.state_dict(), LSTM_MODEL_PATH)

lstm_metadata = {
    "labels": list(mlb.classes_),
    "threshold": float(best_threshold),
    "config": dict(best_config.__dict__),
    "token_to_id": vocabulary.token_to_id,
    "id_to_token": vocabulary.id_to_token,
}
joblib.dump(lstm_metadata, LSTM_METADATA_PATH)

print(f"Modele LSTM sauvegarde : {LSTM_MODEL_PATH}")
print(f"Metadonnees LSTM sauvegardees : {LSTM_METADATA_PATH}")


Modele LSTM sauvegarde : C:\Users\ClémentPERRET\OneDrive - EQUATERRE-VDS\Bureau\Cours\DeepLearning\models\lstm_novelforge.pt
Metadonnees LSTM sauvegardees : C:\Users\ClémentPERRET\OneDrive - EQUATERRE-VDS\Bureau\Cours\DeepLearning\models\lstm_metadata.joblib


## Jalon 7 - Comparaison finale Baseline ML vs Deep Learning

La baseline TF-IDF + regression logistique regularisee du notebook 2 obtient environ **F1 micro = 0.42** et **F1 macro = 0.37** sur le dataset complet avec 34 genres. Elle reste rapide, robuste et adaptee aux textes courts/descriptifs, car beaucoup de genres sont fortement associes a des mots-cles explicites.

Le LSTM execute dans ce notebook utilise un sous-echantillon CPU de **12 000 lignes**, une couche d'embedding, un LSTM bidirectionnel, Adam, EarlyStopping, `pos_weight` pour les classes rares et une recherche de seuil sur validation. Le run courant obtient **F1 micro = 0.172** et **F1 macro = 0.159**, pour environ **392 secondes** de recherche/entrainement. Le rappel est tres eleve sur beaucoup de genres, mais la precision chute fortement : le modele predit trop de labels positifs, ce qui se voit aussi dans la Hamming loss elevee (**0.867**).

Conclusion : pour l'instant, la baseline ML est meilleure en performance et en temps de calcul. Le LSTM reste pertinent pedagogiquement pour modeliser la sequentialite, mais il demanderait plus de tuning pour etre competitif : embeddings pre-entraines, plus d'epochs, meilleur calibrage des seuils par label, regularisation plus fine, ou architectures modernes type CNN/RNN hybride puis Transformers. Pour NovelForge, TF-IDF reste donc la reference solide; le DL fondamental sert a tester si l'ordre des mots apporte un gain au-dela des mots-cles.